#TODO Ainda é necessario tratar os horarios de chamados e tempo de atendimento.

In [0]:
from config import BRONZE_SCHEMA,SILVER_SCHEMA, CATALOG
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
bronze_chamados = spark.table(f"{BRONZE_SCHEMA}.chamados")
bronze_chamados.printSchema()
display(bronze_chamados)

In [0]:
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente','resolvido'])
dq_check_unique('chamados_bronze','unique_id',bronze_chamados,['id_chamado','id_cliente','canal','id_atendente','resolvido','motivo'])

In [0]:
from pyspark.sql.window import Window
win = Window.partitionBy('id_chamado')
duplicados = (
    bronze_chamados
    .withColumn('quantidade', F.count('*').over(win))
    .filter(F.col('quantidade')>1)
)
display(duplicados)
#aparentemente existem linha completamente duplicadas 

In [0]:
display(bronze_chamados.select('motivo').groupBy('motivo').count().orderBy('count'))

In [0]:
silver_chamados= (
    bronze_chamados
    .withColumn(
        'nome_motivo',
        F.lower(F.expr("trim(motivo) COLLATE UNICODE_CI_AI"
        )),
    )
    .drop('motivo')
)
display(silver_chamados.groupBy('nome_motivo').count().orderBy('count'))

In [0]:
#removendo o motivo e utilizando somento o id. afim de normalizar o modelo relacional
silver_motivos = spark.table(f"{SILVER_SCHEMA}.base_motivos")
motivos= (
    silver_motivos.select('nome_motivo','id_motivo')
    .select('id_motivo','nome_motivo')
    .withColumn(
        'nome_motivo',
        F.expr("trim(nome_motivo) COLLATE UNICODE_CI_AI"),
    )
)
#
silver_chamados=(
    silver_chamados
    .join(motivos, on ='nome_motivo', how='left')
    .drop('nome_motivo')
)
display(silver_chamados)

In [0]:
subs={
    '_': ' ',
    '.': '',
    'atend': 'atendimento',
    'chat bot': 'chatbot'
}
parciais =['atendimento']
def string_mapping(df,column,subs:dict,parciais:list):
    import re

    df_mapped = (
        df.withColumn(column, F.lower(F.trim(F.col(f'{column}'))),
                      )
    )
    expressao = F.col(column)
    for original, substituto in subs.items():
        expressao= F.regexp_replace(
            expressao, re.escape(original), substituto
        )
    df_mapped = df_mapped.withColumn(column, expressao)
    for parcial in parciais:
        df_mapped = df_mapped.withColumn(column,
                                     F.regexp_replace(
                                         F.col(column),
                                         rf"{parcial}\S*",
                                         parcial
                                     ))
    return df_mapped

df_ch = string_mapping(silver_chamados,'canal',subs,parciais)
display(df_ch.groupBy('canal').count())


In [0]:
silver_chamados = df_ch

In [0]:
#aqui eu acho que um mapeamento direto e simples é o correto devido as poucas possibilidades
subs ={
    'nao resolvido':'0',
    'nao': '0',
    'não': '0',
    'n':'0',
    '00':'0',
    'sim':'1',
    's': '1',

}
parcias=['1','0']
df_ch = string_mapping(silver_chamados,'resolvido',subs,parciais)
display(df_ch.groupBy('resolvido').count())

In [0]:
df_ch =(df_ch.withColumn('resolvido', 
                 F.col('resolvido').try_cast('int'))
                 .filter(F.col('resolvido').between(0,1) | (F.col('resolvido').isNull()))
)
silver_chamados = df_ch
#Ainda existem valores nulos

In [0]:
#TODO